# Temporal v3 Training Profile Plots

Read `training_profile.jsonl` from a profiler run and plot timing, throughput, and memory.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

RUN_ROOT = Path('D:/TradingML/runtimes/temporal_event_model/v3/profile')
candidates = sorted(RUN_ROOT.glob('*/training_profile.jsonl'), key=lambda p: p.stat().st_mtime, reverse=True)
PROFILE_PATH = candidates[0] if candidates else RUN_ROOT / 'training_profile.jsonl'
print('PROFILE_PATH:', PROFILE_PATH)
rows = []
with PROFILE_PATH.open('r', encoding='utf-8') as handle:
    for line in handle:
        if line.strip():
            rows.append(json.loads(line))
df = pd.DataFrame(rows)
df = df[df.get('phase', '') == 'measure'].reset_index(drop=True) if 'phase' in df else df
display(df.head())
display(df.describe(numeric_only=True).T[['mean', 'std', 'min', 'max']].sort_index())

In [ ]:
timing_cols = [
    'loader_wait_seconds', 'host_to_device_seconds', 'forward_seconds',
    'loss_seconds', 'backward_seconds', 'optimizer_seconds'
]
available = [col for col in timing_cols if col in df]
ax = df[available].plot(figsize=(14, 5), marker='o')
ax.set_title('Per-batch training timing')
ax.set_xlabel('Measured batch')
ax.set_ylabel('Seconds')
ax.grid(True, alpha=0.3)
plt.show()

if 'samples_per_second' in df:
    ax = df['samples_per_second'].plot(figsize=(14, 4), marker='o')
    ax.set_title('Samples per second')
    ax.set_xlabel('Measured batch')
    ax.grid(True, alpha=0.3)
    plt.show()

In [ ]:
loader_cols = [col for col in df.columns if col.startswith('loader/') and col.endswith('_seconds')]
loader_means = df[loader_cols].mean(numeric_only=True).sort_values(ascending=False).head(24)
display(loader_means.to_frame('mean_seconds'))
ax = loader_means.iloc[::-1].plot(kind='barh', figsize=(12, 8))
ax.set_title('Slowest loader stages')
ax.set_xlabel('Mean seconds per batch')
ax.grid(True, axis='x', alpha=0.3)
plt.show()

In [ ]:
memory_cols = [col for col in ['cpu_rss_gib', 'gpu_memory_allocated_gib', 'gpu_memory_reserved_gib', 'gpu_memory_peak_gib'] if col in df]
if memory_cols:
    ax = df[memory_cols].plot(figsize=(14, 5), marker='o')
    ax.set_title('Memory')
    ax.set_xlabel('Measured batch')
    ax.set_ylabel('GiB')
    ax.grid(True, alpha=0.3)
    plt.show()